# Padel Analytics — Interactive Dashboard

This notebook loads already-exported results (`shots.json`, `shots.csv`) and renders a full interactive analytics dashboard.

**You do NOT need to re-run the pipeline.** Just run `main.py` once, then open this notebook.

Sections:
1. Load exported data
2. Match overview cards
3. Shot distribution charts
4. Shot timeline
5. Court heatmaps
6. Rally breakdown
7. Per-player deep-dive
8. Export dashboard PDF

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path('../').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display, HTML

matplotlib.rcParams['figure.dpi'] = 120
plt.style.use('dark_background')

OUTPUTS = ROOT / 'data' / 'outputs'
print(f'Outputs dir: {OUTPUTS}')
print('Setup done ✓')

In [ ]:
import sys, json
from pathlib import Path

ROOT = Path('../').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from IPython.display import display, HTML

matplotlib.rcParams['figure.dpi'] = 120
plt.style.use('dark_background')

OUTPUTS = ROOT / 'data' / 'outputs'
print(f'Outputs dir: {OUTPUTS}')
print('Setup done ✓')

## 1 · Load exported data

In [ ]:
# ── Load shots.json ──────────────────────────────────────────────────────────
json_path = OUTPUTS / 'shots.json'
csv_path  = OUTPUTS / 'shots.csv'

if not json_path.exists():
    print('ERROR: shots.json not found.')
    print('Run:  python main.py --input data/raw/match.mp4')
    raise SystemExit

with open(json_path) as f:
    data = json.load(f)

summary = data['summary']
shots   = data['shots']
meta    = data['meta']

df = pd.DataFrame(shots)

print(f"Exported at   : {meta['exported_at']}")
print(f"Total shots   : {meta['total_shots']}")
print(f"Total players : {meta['total_players']}")
print(f"Duration      : {summary.get('duration_sec', 0):.1f}s")
print(f"Rallies       : {summary.get('total_rallies', 0)}")
display(df.head())

## 2 · Match overview cards

In [ ]:
duration   = summary.get('duration_sec', 0)
tot_shots  = summary.get('total_shots', 0)
tot_rally  = summary.get('total_rallies', 0)
avg_rally  = summary.get('avg_rally_shots', 0)
avg_dur    = summary.get('avg_rally_duration', 0)

CARD_COLOUR = '#1e2a3a'
ACCENT      = '#2196F3'

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
fig.patch.set_facecolor('#111827')

cards = [
    ('Duration',        f'{duration:.0f}s',   '#2196F3'),
    ('Total shots',     str(tot_shots),        '#4CAF50'),
    ('Total rallies',   str(tot_rally),        '#FF9800'),
    ('Avg shots/rally', f'{avg_rally:.1f}',    '#E91E63'),
    ('Avg rally dur',   f'{avg_dur:.1f}s',     '#9C27B0'),
]

for ax, (title, value, colour) in zip(axes, cards):
    ax.set_facecolor(CARD_COLOUR)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0,0),1,1, color=CARD_COLOUR, zorder=0))
    ax.add_patch(plt.Rectangle((0,0.88),1,0.12, color=colour, zorder=1))
    ax.text(0.5, 0.94, title, ha='center', va='center',
            fontsize=9,  color='white', fontweight='bold', zorder=2)
    ax.text(0.5, 0.45, value, ha='center', va='center',
            fontsize=22, color=colour,  fontweight='bold', zorder=2)

plt.suptitle('Match Overview', color='white', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(str(OUTPUTS / 'dashboard_cards.png'),
            dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 3 · Shot distribution

In [ ]:
if df.empty:
    print('No shot data.')
else:
    SHOT_COLOURS = {
        'forehand': '#2196F3',
        'backhand': '#F44336',
        'smash':    '#FF9800',
        'unknown':  '#9E9E9E',
    }

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#111827')

    # ── Left: grouped bar per player ──────────────────────────────────────
    ax = axes[0]
    ax.set_facecolor('#1e2a3a')
    players    = sorted(df['track_id'].unique())
    shot_types = ['forehand', 'backhand', 'smash']
    x          = np.arange(len(players))
    w          = 0.25

    for i, st in enumerate(shot_types):
        vals = [len(df[(df['track_id']==p) & (df['shot_type']==st)]) for p in players]
        bars = ax.bar(x + i*w, vals, w,
                      label=st.capitalize(),
                      color=SHOT_COLOURS[st],
                      edgecolor='#ffffff22')
        for bar, v in zip(bars, vals):
            if v > 0:
                ax.text(bar.get_x()+bar.get_width()/2,
                        bar.get_height()+0.1, str(v),
                        ha='center', va='bottom',
                        fontsize=9, color='white')

    ax.set_xticks(x + w)
    ax.set_xticklabels([f'Player {p}' for p in players], color='white')
    ax.set_ylabel('Shot count', color='white')
    ax.set_title('Shots per Player', color='white', fontsize=12)
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#ffffff22')
    ax.legend(facecolor='#0f3460', labelcolor='white')

    # ── Right: pie chart overall ───────────────────────────────────────────
    ax2 = axes[1]
    ax2.set_facecolor('#1e2a3a')
    type_counts = df['shot_type'].value_counts()
    colours     = [SHOT_COLOURS.get(t, '#9E9E9E') for t in type_counts.index]
    wedges, texts, autotexts = ax2.pie(
        type_counts.values,
        labels=type_counts.index,
        colors=colours,
        autopct='%1.1f%%',
        startangle=90,
        wedgeprops=dict(edgecolor='#ffffff22', linewidth=0.5),
    )
    for text in texts + autotexts:
        text.set_color('white')
    ax2.set_title('Shot Type Distribution', color='white', fontsize=12)

    plt.tight_layout()
    plt.savefig(str(OUTPUTS / 'dashboard_distribution.png'),
                dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 4 · Shot timeline

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(14, 4))
    fig.patch.set_facecolor('#111827')
    ax.set_facecolor('#1e2a3a')

    markers = ['o', 's', '^', 'D']
    players = sorted(df['track_id'].unique())

    for _, row in df.iterrows():
        pidx   = players.index(row['track_id']) % len(markers)
        colour = SHOT_COLOURS.get(row['shot_type'], '#9E9E9E')
        ax.scatter(row['timestamp_sec'], row['track_id'],
                   c=colour, marker=markers[pidx],
                   s=80, edgecolors='#ffffff33', linewidths=0.5, zorder=3)

    # Rally shading from summary
    for rally in summary.get('rallies', []):
        ax.axvspan(rally['start_time_sec'], rally['end_time_sec'],
                   alpha=0.08, color='white', zorder=1)

    ax.set_xlabel('Time (seconds)', color='white')
    ax.set_ylabel('Player ID',      color='white')
    ax.set_title('Shot Timeline',   color='white', fontsize=13)
    ax.set_yticks(players)
    ax.set_yticklabels([f'Player {p}' for p in players], color='white')
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#ffffff22')

    import matplotlib.patches as mpatches
    patches = [mpatches.Patch(color=c, label=t.capitalize())
               for t, c in SHOT_COLOURS.items() if t != 'unknown']
    ax.legend(handles=patches, facecolor='#0f3460',
              labelcolor='white', loc='upper right')

    plt.tight_layout()
    plt.savefig(str(OUTPUTS / 'dashboard_timeline.png'),
                dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 5 · Velocity & Confidence analysis

In [ ]:
if not df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.patch.set_facecolor('#111827')

    for ax in axes:
        ax.set_facecolor('#1e2a3a')
        ax.tick_params(colors='white')
        ax.spines[:].set_color('#ffffff22')

    # Wrist velocity per shot type
    for st, grp in df.groupby('shot_type'):
        axes[0].hist(grp['wrist_velocity'], bins=15, alpha=0.7,
                     label=st.capitalize(),
                     color=SHOT_COLOURS.get(st, '#9E9E9E'),
                     edgecolor='#ffffff22')
    axes[0].set_xlabel('Wrist velocity (px/frame)', color='white')
    axes[0].set_ylabel('Count', color='white')
    axes[0].set_title('Velocity Distribution', color='white', fontsize=11)
    axes[0].legend(facecolor='#0f3460', labelcolor='white', fontsize=8)

    # Forearm angle
    for st, grp in df.groupby('shot_type'):
        axes[1].hist(grp['forearm_angle'], bins=15, alpha=0.7,
                     label=st.capitalize(),
                     color=SHOT_COLOURS.get(st, '#9E9E9E'),
                     edgecolor='#ffffff22')
    axes[1].axvline(0, color='white', linestyle='--', linewidth=1, alpha=0.5)
    axes[1].set_xlabel('Forearm angle (°)', color='white')
    axes[1].set_ylabel('Count', color='white')
    axes[1].set_title('Angle Distribution', color='white', fontsize=11)
    axes[1].legend(facecolor='#0f3460', labelcolor='white', fontsize=8)

    # Confidence
    for st, grp in df.groupby('shot_type'):
        axes[2].hist(grp['confidence'], bins=10, alpha=0.7,
                     label=st.capitalize(),
                     color=SHOT_COLOURS.get(st, '#9E9E9E'),
                     edgecolor='#ffffff22')
    axes[2].set_xlabel('Confidence score', color='white')
    axes[2].set_ylabel('Count', color='white')
    axes[2].set_title('Confidence Distribution', color='white', fontsize=11)
    axes[2].legend(facecolor='#0f3460', labelcolor='white', fontsize=8)

    plt.tight_layout()
    plt.savefig(str(OUTPUTS / 'dashboard_velocity_angle.png'),
                dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.show()

## 6 · Per-player deep-dive

In [ ]:
if not df.empty:
    players = sorted(df['track_id'].unique())
    for pid in players:
        pdf = df[df['track_id'] == pid]
        stats = summary.get('player_stats', {}).get(f'player_{pid}', {})

        print(f'\n── Player {pid} ────────────────────────────────')
        print(f'  Total shots      : {len(pdf)}')
        print(f'  Shot rate/min    : {stats.get("shot_rate_per_min", 0):.1f}')
        counts = pdf['shot_type'].value_counts()
        for st, n in counts.items():
            bar = '█' * n + '░' * max(0, 20-n)
            print(f'  {st:<12} : {bar} {n}')

        if len(pdf) >= 2:
            fig, axes = plt.subplots(1, 3, figsize=(14, 3))
            fig.patch.set_facecolor('#111827')
            fig.suptitle(f'Player {pid} — Detailed Stats',
                         color='white', fontsize=12)

            for ax in axes:
                ax.set_facecolor('#1e2a3a')
                ax.tick_params(colors='white')
                ax.spines[:].set_color('#ffffff22')

            # Shot type counts
            type_c = pdf['shot_type'].value_counts()
            axes[0].bar(type_c.index, type_c.values,
                        color=[SHOT_COLOURS.get(t, '#9E9E9E') for t in type_c.index],
                        edgecolor='#ffffff22')
            axes[0].set_title('Shot types', color='white', fontsize=10)
            axes[0].set_ylabel('Count', color='white')
            for tick in axes[0].get_xticklabels():
                tick.set_color('white')

            # Velocity over time
            axes[1].plot(pdf['timestamp_sec'], pdf['wrist_velocity'],
                         color='#2196F3', linewidth=1, marker='o',
                         markersize=4)
            axes[1].set_title('Wrist velocity over time', color='white', fontsize=10)
            axes[1].set_xlabel('Time (s)', color='white')
            axes[1].set_ylabel('Velocity', color='white')

            # Angle scatter
            for st, grp in pdf.groupby('shot_type'):
                axes[2].scatter(grp['timestamp_sec'], grp['forearm_angle'],
                                label=st.capitalize(),
                                color=SHOT_COLOURS.get(st, '#9E9E9E'),
                                s=50, edgecolors='#ffffff33')
            axes[2].axhline(0, color='white', linestyle='--',
                            linewidth=0.8, alpha=0.5)
            axes[2].set_title('Forearm angle over time', color='white', fontsize=10)
            axes[2].set_xlabel('Time (s)', color='white')
            axes[2].set_ylabel('Angle (°)', color='white')
            axes[2].legend(facecolor='#0f3460', labelcolor='white', fontsize=7)

            plt.tight_layout()
            plt.savefig(str(OUTPUTS / f'dashboard_player{pid}.png'),
                        dpi=150, bbox_inches='tight',
                        facecolor=fig.get_facecolor())
            plt.show()

## 7 · Summary table

In [ ]:
if not df.empty:
    summary_rows = []
    for pid in sorted(df['track_id'].unique()):
        pdf = df[df['track_id'] == pid]
        stats = summary.get('player_stats', {}).get(f'player_{pid}', {})
        row = {
            'Player':          f'Player {pid}',
            'Total':           len(pdf),
            'Forehand':        len(pdf[pdf['shot_type']=='forehand']),
            'Backhand':        len(pdf[pdf['shot_type']=='backhand']),
            'Smash':           len(pdf[pdf['shot_type']=='smash']),
            'Avg velocity':    round(pdf['wrist_velocity'].mean(), 2),
            'Avg confidence':  round(pdf['confidence'].mean(), 3),
            'Shots/min':       stats.get('shot_rate_per_min', 0),
        }
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    display(HTML(summary_df.to_html(index=False,
                                    classes='table',
                                    border=0)))

    # Save as CSV
    summary_df.to_csv(str(OUTPUTS / 'player_summary.csv'), index=False)
    print('Saved → data/outputs/player_summary.csv')